In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

In [2]:
import matplotlib
from matplotlib import pyplot as plt
matplotlib.style.use('seaborn')

In [ ]:
data

In [ ]:
n = 20
delta = 0.25

In [ ]:
datat = data[1:n +1]
datat_1 = data[0:n]

In [ ]:
plt.scatter(datat, datat_1)

In [ ]:
Sx = np.sum(datat_1)
Sy = np.sum(datat)
Sxx = np.sum(datat_1 * datat_1)
Syy = np.sum(datat * datat)
Sxy = np.sum(datat * datat_1)

In [ ]:
a = (n * Sxy - Sx * Sy)/(n* Sxx - Sx * Sx)
b = (Sy - a * Sx)/n
sde = np.sqrt((n * Syy - Sy * Sy - a * (n * Sxy - Sx * Sy))/(n * (n-2)))

In [ ]:
lambd = -np.log(a)/ delta
mu = b/(1-a)
sigma = sde * np.sqrt(-2*np.log(a)/(delta*(1-a*a)))

#    Pairs trading - a modified version

Simple moving average, weighted moving average, money flow, relative strenght index

In [ ]:
# Data utilities
def index_by_date(df):
    try:
        df['date'] = pd.to_datetime(df['date'], format='%d/%m/%Y')
    except:
        df['date'] = pd.to_datetime(df['date'], format='%Y/%m/%d')
    df = df.set_index('date')
    
    return df

def adjust_returns(spot_return, decay):
    spot_momentum = spot_return.ewm(com=decay / (1.0 - decay), adjust=False).mean()
    adj_return = spot_return - spot_momentum
    
    return adj_return

In [ ]:
path = '/Users/ana/Quant research/data/'
eurusd = pd.read_csv('{}mktdata_EURUSD.csv'.format(path))
usdchf = pd.read_csv('{}mktdata_USDCHF.csv'.format(path))
eurchf = pd.read_csv('{}mktdata_EURCHF.csv'.format(path))
eurusd = index_by_date(eurusd)
usdchf = index_by_date(usdchf)
eurchf = index_by_date(eurchf)

In [ ]:
eur = eurusd['Spot']
chf = usdchf['Spot']
eurchf = eurchf['Spot']

In [ ]:
eur = np.log(eur).diff()
chf = - np.log(chf).diff()
eur = eur.dropna()
chf = chf.dropna()
eur.plot()
chf.plot()

In [ ]:
#eurchf = np.log(eurchf).diff()
#eurchf.cumsum().plot()

In [ ]:
eur_arr = eur.values
eur_arr = eur_arr.reshape(-1, 1) 

In [ ]:
chf_arr = chf.values
chf_arr = chf_arr.reshape(-1, 1) 

In [ ]:
plt.scatter(eur_arr, chf_arr)

In [ ]:
lr = LinearRegression()

In [ ]:
lr.fit(eur_arr, chf_arr)

In [ ]:
a = lr.coef_[0]
b = lr.intercept_

So the beta in the Eq (1) is

# Pairs trading (https://towardsdatascience.com/quant-trading-an-introduction-to-pairs-trading-5ce50c03177e)

In [ ]:
data = pd.read_csv('{}pairs_data.csv'.format(path))
data = index_by_date(data)

In [ ]:
import seaborn
import statsmodels
import statsmodels.api as sm
from statsmodels.tsa.stattools import coint

In [ ]:
def find_cointegrated_pairs(data):
    n = data.shape[1]
    score_matrix = np.zeros((n,n))
    pvalue_matrix = np.ones((n,n))
    keys = data.keys()
    pairs = []
    for i in range(n):
        for j in range(i+1, n):
            S1 = data[keys[i]]
            S2 = data[keys[j]]
            result = coint(S1, S2)
            score = result[0]
            pvalue = result[1]
            score_matrix[i, j] = score
            pvalue_matrix[i, j]= pvalue
            if pvalue < 0.01:
                pairs.append((keys[i], keys[j]))
    return score_matrix, pvalue_matrix, pairs

In [ ]:
score, pvalue, pairs = find_cointegrated_pairs(data)

In [ ]:
seaborn.heatmap(pvalue, xticklabels = data.columns, yticklabels =data.columns, cmap = 'plasma', mask = (pvalue >= 0.01))

It didn't find cointegration for USDCHF, which I expected with EURUSD. Let's see what happens if I add the inverse, so CHFUSD

In [ ]:
data['CHFUSD'] = 1/ data['USDCHF']

In [ ]:
plt.scatter(-np.log(data['USDCHF']).diff(), np.log(data['EURUSD']).diff())

In [ ]:
plt.scatter(np.log(data['EURUSD']).diff(), -np.log(data['USDCHF']).diff())

In [ ]:
plt.scatter(np.log(data['USDSEK']).diff(), np.log(data['USDCAD']).diff())

In [ ]:
score, pvalue, pairs = find_cointegrated_pairs(data)
seaborn.heatmap(pvalue, xticklabels = data.columns, yticklabels =data.columns, cmap = 'plasma', mask = (pvalue >= 0.01))

In [ ]:
coint_pairs = [('Copper', 'USDCAD'), ('USDCAD', 'USDSEK'), ('VIX', 'CHFUSD'), ('EURUSD', 'USDCHF')]

In [ ]:
for i in coint_pairs:
    S1 = data[i[0]]
    S2 = data[i[1]]
    score, pvalue, _ = coint(S1, S2)
    print(pvalue)

In [ ]:
coint_pairs

In [ ]:
spread = data['USDCAD'] - data['USDSEK']
spread.plot(label = 'Spread')
plt.axhline(spread.mean(),c = 'r')

In [ ]:
#normalize stock prices
def zscore(stocks):
    return (stocks - stocks.mean())/np.std(stocks)
zscore(spread).plot()
plt.axhline(zscore(spread).mean(),c = 'b')
plt.axhline(1.0, c = 'r', ls = '--')
plt.axhline(-1.0, c = 'r', ls = '--')

In [ ]:
spread_mavg1 = spread.rolling(1).mean()
spread_mavg20 = spread.rolling(20).mean()
spread_std20 = spread.rolling(20).std()
zscore_20_1 = (spread_mavg1 - spread_mavg20)/spread_std20
zscore_20_1.plot(label = 'Rolling 20 day z score')
plt.axhline(0,color = 'black')
plt.axhline(1, c ='r',ls='--')
plt.axhline(-1, c ='r',ls='--')